In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
import time

from tqdm import tqdm

import numpy as np
import pandas as pd

import hyperopt.hp as hp

from sklearn.model_selection import train_test_split

from catboost import CatBoostClassifier

from sklift.datasets import fetch_x5
from sklift.models import ClassTransformation, TwoModels
from sklift.metrics import qini_auc_score, uplift_at_k

from causalml.inference.tree import UpliftTreeClassifier as UpliftTreeClassifierCM

from upninja.dml.uplift_tree_dml import UpliftTreeRegressorDML
from upninja.tune.selection import UpliftTune

import matplotlib.pyplot as plt
import seaborn as sns

Failed to import duecredit due to No module named 'duecredit'


In [3]:
SEED = 8

In [4]:
%%time

dataset = fetch_x5()
dataset.data.keys()

CPU times: user 26.9 s, sys: 5.94 s, total: 32.8 s
Wall time: 35.3 s


dict_keys(['clients', 'train', 'purchases'])

In [5]:
%%time

print(f'Dataset type: {type(dataset)}\n')
print(f'Dataset features shape: {dataset.data['clients'].shape}')
print(f'Dataset features shape: {dataset.data['train'].shape}')
print(f'Dataset target shape: {dataset.target.shape}')
print(f'Dataset treatment shape: {dataset.treatment.shape}')

Dataset type: <class 'sklearn.utils._bunch.Bunch'>

Dataset features shape: (400162, 5)
Dataset features shape: (200039, 1)
Dataset target shape: (200039,)
Dataset treatment shape: (200039,)
CPU times: user 72 μs, sys: 88 μs, total: 160 μs
Wall time: 157 μs


In [6]:
%%time

# Извлечение данных
df_clients = dataset.data['clients'].set_index('client_id')
df_train = pd.concat([dataset.data['train'], dataset.treatment , dataset.target], axis=1).set_index('client_id')
indices_test = pd.Index(set(df_clients.index) - set(df_train.index))

# Извлечение признаков
df_features = df_clients.copy()
df_features['first_issue_time'] = \
    (pd.to_datetime(df_features['first_issue_date'])
     - pd.Timestamp('1970-01-01')) // pd.Timedelta('1s')
df_features['first_redeem_time'] = \
    (pd.to_datetime(df_features['first_redeem_date'])
     - pd.Timestamp('1970-01-01')) // pd.Timedelta('1s')
df_features['issue_redeem_delay'] = df_features['first_redeem_time'] \
    - df_features['first_issue_time']
df_features = df_features.drop(['first_issue_date', 'first_redeem_date'], axis=1)

indices_learn, indices_valid = train_test_split(df_train.index, test_size=0.3, random_state=SEED)

CPU times: user 186 ms, sys: 38.8 ms, total: 225 ms
Wall time: 224 ms


In [7]:
%%time

X_train = df_features.loc[indices_learn, :]
y_train = df_train.loc[indices_learn, 'target']
treat_train = df_train.loc[indices_learn, 'treatment_flg']

X_val = df_features.loc[indices_valid, :]
y_val = df_train.loc[indices_valid, 'target']
treat_val =  df_train.loc[indices_valid, 'treatment_flg']

X_train_full = df_features.loc[df_train.index, :]
y_train_full = df_train.loc[:, 'target']
treat_train_full = df_train.loc[:, 'treatment_flg']

X_test = df_features.loc[indices_test, :]

X_train['gender'] = X_train['gender'].map({'F': 0, 'U': -1, 'M': 1})
X_val['gender'] = X_val['gender'].map({'F': 0, 'U': -1, 'M': 1})
X_test['gender'] = X_test['gender'].map({'F': 0, 'U': -1, 'M': 1})

X_train.fillna(-1.0, inplace=True)
X_val.fillna(-1.0, inplace=True)
X_test.fillna(-1.0, inplace=True)

cat_features = ['gender']

CPU times: user 250 ms, sys: 5.51 ms, total: 256 ms
Wall time: 255 ms


# ✅ Train models

## ⭐ Class Transformation

In [8]:
%%time

cb_params = cb_hp_space = {
    'iterations': hp.uniformint('iterations', 100, 2500),
    'depth': hp.uniformint('depth', 2, 10),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
    'l2_leaf_reg': hp.uniform('l2_leaf_reg', 1, 10),
    'verbose': False,
    'thread_count': -1,
    'random_state': SEED
}

cb_tune = UpliftTune(
    base_model_class=CatBoostClassifier,
    uplift_model_class=ClassTransformation,
    data=X_train,
    target=y_train,
    treatment=treat_train,
    space=cb_params,
    verbose=True,
    max_evals=10
)

t_time_start = time.process_time_ns()
cb_tuned = cb_tune.tune()['best_params']
t_time_end = time.process_time_ns()

100%|███████| 10/10 [02:51<00:00, 17.13s/trial, best loss: -0.04954288228464684]
Optimization completed. Best score: 0.0495
Best parameters: {'depth': 2, 'iterations': 2166, 'l2_leaf_reg': 6.474242459540229, 'learning_rate': 0.06708005741019658, 'random_state': 8, 'thread_count': -1, 'verbose': False}
CPU times: user 20min 21s, sys: 2min 29s, total: 22min 51s
Wall time: 2min 51s


In [9]:
%%time

ct = ClassTransformation(CatBoostClassifier(**cb_tuned))
f_time_start = time.process_time_ns()
ct = ct.fit(X_train, y_train, treat_train, estimator_fit_params={'cat_features': cat_features})
f_time_end = time.process_time_ns()

ct_uplift = ct.predict(X_val)

CPU times: user 3min, sys: 10.9 s, total: 3min 11s
Wall time: 21.4 s


## ⭐ DML Tree

In [10]:
%%time

space = {
    'min_samples': hp.uniformint('min_samples', 20, 500),
    'max_depth': hp.uniformint('max_depth', 3, 20),
    'random_state': SEED
}

dml_tune = UpliftTune(
    uplift_model_class=UpliftTreeRegressorDML,
    data=X_train,
    target=y_train,
    treatment=treat_train,
    space=space,
    verbose=True,
    max_evals=10
)

t_time_start = time.process_time_ns()
dml_tuned = dml_tune.tune()['best_params']
t_time_end = time.process_time_ns()

100%|███████| 10/10 [00:09<00:00,  1.07trial/s, best loss: -0.05889112202442145]
Optimization completed. Best score: 0.0589
Best parameters: {'max_depth': 14, 'min_samples': 411, 'random_state': 8}
CPU times: user 9.05 s, sys: 169 ms, total: 9.21 s
Wall time: 9.33 s


In [11]:
%%time

dml = UpliftTreeRegressorDML(**dml_tuned)

f_time_start = time.process_time_ns()
dml = dml.fit(X_train, y_train, treat_train)
f_time_end = time.process_time_ns()

dml_uplift = dml.predict(X_val)

CPU times: user 539 ms, sys: 7.53 ms, total: 547 ms
Wall time: 550 ms


# ✅ Case

Оценим эффективность назначения коммуникаций при помощи uplift-моделей.

In [12]:
def build_no_horizon_business_case(
    base_df,
    uplift_col,
    quantiles=(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0),
    sms_cost=8,
    check_col='avg_purchase_sum'
):
    """
    Бизнес-кейс без горизонтов:
    - сортируем клиентов по uplift-score;
    - Top N% считаются накопительно сверху скоринга;
    - конверсия считается по исходному target Retail Hero;
    - денежная оценка использует исторический средний чек как proxy;
    - ОП на клиента = uplift конверсии * средний чек - стоимость SMS;
    - Total increment = ОП на клиента * N клиентов.
    """

    df = base_df.copy()

    df_sorted = (
        df
        .sort_values(by=uplift_col, ascending=False)
        .reset_index(drop=True)
    )

    total_n = len(df_sorted)
    rows = []

    for q in quantiles:
        cutoff = int(total_n * q)
        subset = df_sorted.iloc[:cutoff].copy()

        treat = subset[subset['treatment_flg'] == 1]
        control = subset[subset['treatment_flg'] == 0]

        conv_treat = treat['target'].mean()
        conv_control = control['target'].mean()
        uplift_conv = conv_treat - conv_control

        # Средний чек как proxy стоимости покупки
        avg_check = subset.loc[subset['target'] == 1, check_col].mean()

        if pd.isna(avg_check):
            avg_check = subset[check_col].mean()

        op_per_client = uplift_conv * avg_check - sms_cost
        total_increment = op_per_client * len(subset)

        rows.append({
            'Модель': uplift_col,
            'Доля базы': f'Top {int(q * 100)}%',
            'Доля базы sort': int(q * 100),
            'Кол-во наблюдений': len(subset),

            'Конверсия в КГ': conv_control,
            'Конверсия в ТГ': conv_treat,
            'Инкремент конверсии': uplift_conv,

            'Средний чек proxy, руб': avg_check,

            'Инкремент ОП на клиента, руб': op_per_client,
            'Total increment, руб': total_increment,
        })

    return pd.DataFrame(rows)

In [13]:
def make_no_horizon_slide_table(long_df):
    df = long_df.copy()

    df['Конверсия в КГ'] = (
        (df['Конверсия в КГ'] * 100)
        .round(1)
        .astype(str)
        .str.replace('.', ',', regex=False)
        + '%'
    )

    df['Конверсия в ТГ'] = (
        (df['Конверсия в ТГ'] * 100)
        .round(1)
        .astype(str)
        .str.replace('.', ',', regex=False)
        + '% '
        + df['Инкремент конверсии'].apply(
            lambda x: f"(+{x * 100:.1f}%)" if x >= 0 else f"({x * 100:.1f}%)"
        ).str.replace('.', ',', regex=False)
    )

    df['Средний чек proxy, руб'] = df['Средний чек proxy, руб'].round(1)
    df['Инкремент ОП на клиента, руб'] = df['Инкремент ОП на клиента, руб'].round(1)
    df['Total increment, руб'] = df['Total increment, руб'].round(0).astype(int)

    final = df[[
        'Доля базы',
        'Кол-во наблюдений',
        'Конверсия в КГ',
        'Конверсия в ТГ',
        'Средний чек proxy, руб',
        'Инкремент ОП на клиента, руб',
        'Total increment, руб'
    ]].copy()

    final['_sort'] = final['Доля базы'].str.extract(r'(\d+)').astype(int)
    final = final.sort_values('_sort').drop(columns='_sort').reset_index(drop=True)

    return final

In [33]:
def build_uplift_bin_business_case(
    base_df,
    uplift_col,
    n_bins=10,
    sms_cost=8,
    check_col='avg_purchase_sum'
):
    """
    Считает бизнес-эффект НЕ накопительно Top N%,
    а по отдельным uplift-бинам:
    Top 0-10%, 10-20%, ..., 90-100%.

    Это нужно для поиска отсечки:
    до какого uplift-сегмента коммуникация еще окупается.
    """

    df = base_df.copy()

    df_sorted = (
        df
        .sort_values(by=uplift_col, ascending=False)
        .reset_index(drop=True)
    )

    df_sorted['bin_id'] = pd.qcut(
        df_sorted.index + 1,
        q=n_bins,
        labels=False
    )

    rows = []

    for bin_id in range(n_bins):
        subset = df_sorted[df_sorted['bin_id'] == bin_id].copy()

        left = int(bin_id * 100 / n_bins)
        right = int((bin_id + 1) * 100 / n_bins)

        treat = subset[subset['treatment_flg'] == 1]
        control = subset[subset['treatment_flg'] == 0]

        conv_treat = treat['target'].mean()
        conv_control = control['target'].mean()
        uplift_conv = conv_treat - conv_control

        avg_check = subset.loc[subset['target'] == 1, check_col].mean()

        if pd.isna(avg_check):
            avg_check = subset[check_col].mean()

        op_per_client = uplift_conv * avg_check - sms_cost
        total_increment = op_per_client * len(subset)

        rows.append({
            'Модель': uplift_col,
            'Сегмент': f'Top {left}-{right}%',
            'bin_id': bin_id,
            'Кол-во наблюдений': len(subset),

            'Конверсия в КГ': conv_control,
            'Конверсия в ТГ': conv_treat,
            'Инкремент конверсии': uplift_conv,

            'Средний чек proxy, руб': avg_check,
            'Стоимость контакта, руб': sms_cost,

            'ОП на клиента, руб': op_per_client,
            'Total increment, руб': total_increment,

            'Решение': 'отправлять' if op_per_client > 0 else 'не отправлять'
        })

    return pd.DataFrame(rows)

In [14]:
def get_best_strategy(case_df):
    strategy = case_df.copy()

    strategy = strategy[
        strategy['Инкремент ОП на клиента, руб'] > 0
    ].copy()

    best = (
        strategy
        .sort_values('Total increment, руб', ascending=False)
        .head(1)
        .copy()
    )

    best['Конверсия в КГ'] = (best['Конверсия в КГ'] * 100).round(1)
    best['Конверсия в ТГ'] = (best['Конверсия в ТГ'] * 100).round(1)
    best['Инкремент конверсии'] = (best['Инкремент конверсии'] * 100).round(1)
    best['Средний чек proxy, руб'] = best['Средний чек proxy, руб'].round(1)
    best['Инкремент ОП на клиента, руб'] = best['Инкремент ОП на клиента, руб'].round(1)
    best['Total increment, руб'] = best['Total increment, руб'].round(0).astype(int)

    return best[[
        'Модель',
        'Доля базы',
        'Кол-во наблюдений',
        'Конверсия в КГ',
        'Конверсия в ТГ',
        'Инкремент конверсии',
        'Средний чек proxy, руб',
        'Инкремент ОП на клиента, руб',
        'Total increment, руб'
    ]]

In [15]:
def sensitivity_by_sms_cost(base_case_df, sms_costs=(3, 5, 8, 10, 15)):
    rows = []

    for cost in sms_costs:
        tmp = base_case_df.copy()

        tmp['Инкремент ОП на клиента, руб'] = (
            tmp['Инкремент конверсии'] * tmp['Средний чек proxy, руб'] - cost
        )
        tmp['Total increment, руб'] = (
            tmp['Инкремент ОП на клиента, руб'] * tmp['Кол-во наблюдений']
        )

        best = (
            tmp[tmp['Инкремент ОП на клиента, руб'] > 0]
            .sort_values('Total increment, руб', ascending=False)
            .head(1)
        )

        if len(best) > 0:
            rows.append({
                'Стоимость SMS, руб': cost,
                'Оптимальная доля базы': best['Доля базы'].iloc[0],
                'Кол-во наблюдений': best['Кол-во наблюдений'].iloc[0],
                'ОП на клиента, руб': round(best['Инкремент ОП на клиента, руб'].iloc[0], 2),
                'Total increment, руб': round(best['Total increment, руб'].iloc[0], 0),
            })

    return pd.DataFrame(rows)

In [16]:
def budget_strategy(base_df, case_df, budgets=(50_000, 100_000, 200_000, 500_000), sms_cost=8):
    rows = []
    total_base = len(base_df)

    for budget in budgets:
        n_clients = int(budget // sms_cost)
        share = n_clients / total_base

        # ищем ближайший доступный Top N%
        tmp = case_df.copy()
        tmp['share'] = tmp['Доля базы'].str.extract(r'(\d+)').astype(int) / 100
        tmp['diff'] = (tmp['share'] - share).abs()

        row = tmp.sort_values('diff').head(1).iloc[0]

        rows.append({
            'Бюджет, руб': budget,
            'Макс. клиентов по бюджету': n_clients,
            'Ближайшая доля базы': row['Доля базы'],
            'ОП на клиента, руб': round(row['Инкремент ОП на клиента, руб'], 2),
            'Ожидаемый total increment, руб': round(row['Total increment, руб'], 0),
        })

    return pd.DataFrame(rows)

In [17]:
X_val['ct_uplift'] = ct_uplift
X_val['dml_uplift'] = dml_uplift
X_val = pd.concat([X_val, treat_val, y_val], axis=1)

In [18]:
X_val = X_val.merge(dataset.data['purchases'][['client_id', 'purchase_sum', 'transaction_datetime']], how='left', on=['client_id'])

In [19]:
base_val = df_features.loc[indices_valid, :].copy().reset_index()

if 'client_id' not in base_val.columns:
    base_val = base_val.rename(columns={base_val.columns[0]: 'client_id'})

base_val['ct_uplift'] = ct_uplift
base_val['dml_uplift'] = dml_uplift

base_val = base_val.merge(
    treat_val.rename('treatment_flg').reset_index(),
    on='client_id',
    how='left'
)

base_val = base_val.merge(
    y_val.rename('target').reset_index(),
    on='client_id',
    how='left'
)

base_val = base_val.drop_duplicates(subset=['client_id']).copy()

print(base_val.shape)

(60012, 10)


In [20]:
# Исторический средний чек по клиенту
# purchases — это история покупок, используем ее только как proxy для денежной оценки

purchase_proxy = (
    dataset.data['purchases']
    .groupby('client_id')
    .agg(
        avg_purchase_sum=('purchase_sum', 'mean'),
        total_purchase_sum=('purchase_sum', 'sum'),
        purchase_cnt=('purchase_sum', 'count')
    )
    .reset_index()
)

base_val = base_val.merge(
    purchase_proxy,
    on='client_id',
    how='left'
)

base_val['avg_purchase_sum'] = base_val['avg_purchase_sum'].fillna(
    base_val['avg_purchase_sum'].median()
)

base_val['total_purchase_sum'] = base_val['total_purchase_sum'].fillna(0)
base_val['purchase_cnt'] = base_val['purchase_cnt'].fillna(0)

base_val.head()

,client_id,age,gender,first_issue_time,first_redeem_time,issue_redeem_delay,ct_uplift,dml_uplift,treatment_flg,target,avg_purchase_sum,total_purchase_sum,purchase_cnt
0,e2f4387414,45,M,1519842117,1.525813e+09,5971259.0,0.045889,0.047158,0,1,379.857703,28109.47,74
1,8098ef0f15,18,F,1515614436,NaN,NaN,-0.012391,0.074007,0,0,270.678333,8120.35,30
2,79c9d3a1a8,59,F,1499945782,1.501761e+09,1814782.0,0.029651,0.028702,1,1,577.834895,109788.63,190
3,a299331961,34,U,1507227952,1.514053e+09,6825460.0,0.041730,0.028702,1,1,535.115479,100601.71,188
4,f5df2493fa,60,F,1499508384,1.509122e+09,9613778.0,0.033262,0.028702,1,1,385.758589,62878.65,163


In [32]:
dml_case = build_no_horizon_business_case(
    base_df=base_val,
    uplift_col='dml_uplift',
    quantiles=(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0),
    sms_cost=10,
    check_col='avg_purchase_sum'
)

dml_slide_table = make_no_horizon_slide_table(dml_case)

dml_slide_table

,Доля базы,Кол-во наблюдений,Конверсия в КГ,Конверсия в ТГ,"Средний чек proxy, руб","Инкремент ОП на клиента, руб","Total increment, руб"
0,Top 10%,6001,"58,9%","68,8% (+9,8%)",544.9,43.6,261609
1,Top 20%,12002,"62,0%","68,7% (+6,7%)",541.1,26.5,317724
2,Top 30%,18003,"61,6%","67,3% (+5,7%)",557.3,21.8,392925
3,Top 40%,24004,"61,4%","66,7% (+5,3%)",580.6,20.7,497001
4,Top 50%,30006,"61,7%","66,9% (+5,2%)",602.0,21.2,634798
5,Top 60%,36007,"61,9%","66,9% (+5,0%)",618.7,20.6,742988
6,Top 70%,42008,"61,8%","66,6% (+4,7%)",621.4,19.4,814150
7,Top 80%,48009,"61,3%","65,5% (+4,1%)",619.6,15.7,754168
8,Top 90%,54010,"61,2%","65,0% (+3,8%)",626.7,14.0,754757
9,Top 100%,60012,"60,0%","63,7% (+3,7%)",626.1,13.2,790538


In [29]:
ct_case = build_no_horizon_business_case(
    base_df=base_val,
    uplift_col='ct_uplift',
    quantiles=(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0),
    sms_cost=8,
    check_col='avg_purchase_sum'
)

ct_slide_table = make_no_horizon_slide_table(ct_case)

ct_slide_table

,Доля базы,Кол-во наблюдений,Конверсия в КГ,Конверсия в ТГ,"Средний чек proxy, руб","Инкремент ОП на клиента, руб","Total increment, руб"
0,Top 10%,6001,"64,6%","74,5% (+9,9%)",519.0,43.4,260173
1,Top 20%,12002,"63,4%","70,9% (+7,4%)",547.5,32.8,393163
2,Top 30%,18003,"63,0%","69,7% (+6,7%)",571.9,30.1,542414
3,Top 40%,24004,"62,6%","68,4% (+5,9%)",588.3,26.5,634920
4,Top 50%,30006,"62,5%","67,8% (+5,3%)",603.2,24.2,726044
5,Top 60%,36007,"62,2%","67,2% (+5,1%)",612.4,23.0,828401
6,Top 70%,42008,"61,7%","66,5% (+4,8%)",619.2,21.7,910950
7,Top 80%,48009,"61,5%","65,7% (+4,3%)",625.6,18.7,895986
8,Top 90%,54010,"60,7%","64,8% (+4,0%)",628.8,17.4,939972
9,Top 100%,60012,"60,0%","63,7% (+3,7%)",626.1,15.2,910562


In [30]:
dml_best_strategy = get_best_strategy(dml_case)
ct_best_strategy = get_best_strategy(ct_case)

display(dml_best_strategy)
display(ct_best_strategy)

,Модель,Доля базы,Кол-во наблюдений,Конверсия в КГ,Конверсия в ТГ,Инкремент конверсии,"Средний чек proxy, руб","Инкремент ОП на клиента, руб","Total increment, руб"
9,dml_uplift,Top 100%,60012,60.0,63.7,3.7,626.1,15.2,910562


,Модель,Доля базы,Кол-во наблюдений,Конверсия в КГ,Конверсия в ТГ,Инкремент конверсии,"Средний чек proxy, руб","Инкремент ОП на клиента, руб","Total increment, руб"
8,ct_uplift,Top 90%,54010,60.7,64.8,4.0,628.8,17.4,939972


In [24]:
dml_sensitivity = sensitivity_by_sms_cost(dml_case)
ct_sensitivity = sensitivity_by_sms_cost(ct_case)

dml_sensitivity

,"Стоимость SMS, руб",Оптимальная доля базы,Кол-во наблюдений,"ОП на клиента, руб","Total increment, руб"
0,3,Top 100%,60012,20.17,1210622.0
1,5,Top 100%,60012,18.17,1090598.0
2,8,Top 100%,60012,15.17,910562.0
3,10,Top 70%,42008,19.38,814150.0
4,15,Top 70%,42008,14.38,604110.0


In [25]:
ct_sensitivity

,"Стоимость SMS, руб",Оптимальная доля базы,Кол-во наблюдений,"ОП на клиента, руб","Total increment, руб"
0,3,Top 100%,60012,20.17,1210622.0
1,5,Top 90%,54010,20.40,1102002.0
2,8,Top 90%,54010,17.40,939972.0
3,10,Top 90%,54010,15.40,831952.0
4,15,Top 70%,42008,14.69,616894.0


In [26]:
dml_budget_strategy = budget_strategy(base_val, dml_case, sms_cost=8)
dml_budget_strategy

,"Бюджет, руб",Макс. клиентов по бюджету,Ближайшая доля базы,"ОП на клиента, руб","Ожидаемый total increment, руб"
0,50000,6250,Top 10%,48.59,291614.0
1,100000,12500,Top 20%,31.47,377734.0
2,200000,25000,Top 40%,25.70,617021.0
3,500000,62500,Top 100%,18.17,1090598.0


In [27]:
ct_budget_strategy = budget_strategy(base_val, ct_case, sms_cost=8)
ct_budget_strategy

,"Бюджет, руб",Макс. клиентов по бюджету,Ближайшая доля базы,"ОП на клиента, руб","Ожидаемый total increment, руб"
0,50000,6250,Top 10%,46.35,278176.0
1,100000,12500,Top 20%,35.76,429169.0
2,200000,25000,Top 40%,29.45,706932.0
3,500000,62500,Top 100%,18.17,1090598.0


In [34]:
dml_bins_8 = build_uplift_bin_business_case(
    base_df=base_val,
    uplift_col='dml_uplift',
    n_bins=10,
    sms_cost=8,
    check_col='avg_purchase_sum'
)

ct_bins_8 = build_uplift_bin_business_case(
    base_df=base_val,
    uplift_col='ct_uplift',
    n_bins=10,
    sms_cost=8,
    check_col='avg_purchase_sum'
)

dml_bins_16 = build_uplift_bin_business_case(
    base_df=base_val,
    uplift_col='dml_uplift',
    n_bins=10,
    sms_cost=16,
    check_col='avg_purchase_sum'
)

ct_bins_16 = build_uplift_bin_business_case(
    base_df=base_val,
    uplift_col='ct_uplift',
    n_bins=10,
    sms_cost=16,
    check_col='avg_purchase_sum'
)

display(dml_bins_8)
display(dml_bins_16)

,Модель,Сегмент,bin_id,Кол-во наблюдений,Конверсия в КГ,Конверсия в ТГ,Инкремент конверсии,"Средний чек proxy, руб","Стоимость контакта, руб","ОП на клиента, руб","Total increment, руб",Решение
0,dml_uplift,Top 0-10%,0,6002,0.589021,0.687597,0.098575,544.864674,8,45.710227,274352.783318,отправлять
1,dml_uplift,Top 10-20%,1,6001,0.647727,0.685942,0.038215,537.568315,8,12.543120,75271.261940,отправлять
2,dml_uplift,Top 20-30%,2,6001,0.608140,0.643659,0.035519,590.928158,8,12.989328,77948.956830,отправлять
3,dml_uplift,Top 30-40%,3,6001,0.609894,0.649381,0.039487,652.703580,8,17.773506,106658.807910,отправлять
4,dml_uplift,Top 40-50%,4,6001,0.628533,0.676131,0.047598,685.550372,8,24.630663,147808.610352,отправлять
5,dml_uplift,Top 50-60%,5,6001,0.628801,0.667221,0.038420,702.081965,8,18.974137,113863.793741,отправлять
6,dml_uplift,Top 60-70%,6,6001,0.615259,0.648456,0.033197,637.872476,8,13.175695,79067.345001,отправлять
7,dml_uplift,Top 70-80%,7,6001,0.576633,0.574475,-0.002157,605.631166,8,-9.306629,-55849.077740,не отправлять
8,dml_uplift,Top 80-90%,8,6001,0.603574,0.615307,0.011733,685.179441,8,0.039456,236.772899,отправлять
9,dml_uplift,Top 90-100%,9,6002,0.495463,0.513374,0.017911,619.463011,8,3.095266,18577.783728,отправлять


,Модель,Сегмент,bin_id,Кол-во наблюдений,Конверсия в КГ,Конверсия в ТГ,Инкремент конверсии,"Средний чек proxy, руб","Стоимость контакта, руб","ОП на клиента, руб","Total increment, руб",Решение
0,dml_uplift,Top 0-10%,0,6002,0.589021,0.687597,0.098575,544.864674,16,37.710227,226336.783318,отправлять
1,dml_uplift,Top 10-20%,1,6001,0.647727,0.685942,0.038215,537.568315,16,4.543120,27263.261940,отправлять
2,dml_uplift,Top 20-30%,2,6001,0.608140,0.643659,0.035519,590.928158,16,4.989328,29940.956830,отправлять
3,dml_uplift,Top 30-40%,3,6001,0.609894,0.649381,0.039487,652.703580,16,9.773506,58650.807910,отправлять
4,dml_uplift,Top 40-50%,4,6001,0.628533,0.676131,0.047598,685.550372,16,16.630663,99800.610352,отправлять
5,dml_uplift,Top 50-60%,5,6001,0.628801,0.667221,0.038420,702.081965,16,10.974137,65855.793741,отправлять
6,dml_uplift,Top 60-70%,6,6001,0.615259,0.648456,0.033197,637.872476,16,5.175695,31059.345001,отправлять
7,dml_uplift,Top 70-80%,7,6001,0.576633,0.574475,-0.002157,605.631166,16,-17.306629,-103857.077740,не отправлять
8,dml_uplift,Top 80-90%,8,6001,0.603574,0.615307,0.011733,685.179441,16,-7.960544,-47771.227101,не отправлять
9,dml_uplift,Top 90-100%,9,6002,0.495463,0.513374,0.017911,619.463011,16,-4.904734,-29438.216272,не отправлять


In [36]:
dml_bins_8.to_pickle('res.pkl')